# 🚀 Dual-Branch YOLOv9t for Snow Pole Detection - DESKTOP VERSION

## Overview
This notebook implements a **dual-branch architecture** for LiDAR-based object detection.

**Paper Reference**: Yang et al., "Towards Generalized Range-View LiDAR Segmentation in Adverse Weather" (2025) - [arXiv:2506.08979](https://arxiv.org/abs/2506.08979)

In [2]:
# INSTALL DEPENDENCIES (if needed)
# Uncomment if you need to install packages:
# !pip install ultralytics torch torchvision opencv-python numpy matplotlib pyyaml tqdm

In [1]:
# IMPORTS
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Core imports
import torch
from pathlib import Path
import shutil
import numpy as np
import yaml
from tqdm import tqdm

# OpenCV import with error handling
try:
    import cv2
except ImportError:
    print("OpenCV not found. Installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python"])
    import cv2

# Ultralytics import with error handling
try:
    from ultralytics import YOLO
except ImportError:
    print("Ultralytics not found. Installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    from ultralytics import YOLO

# System info
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"OpenCV version: {cv2.__version__}")
print("-" * 60)

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
PyTorch version: 2.8.0+cpu
CUDA available: False
OpenCV version: 4.12.0
------------------------------------------------------------


In [4]:
# PATHS CONFIGURATION - DESKTOP VERSION
# Update this to your local dataset path
BASE_PATH = Path(r"C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset")

# 3-channel comb images (reflectance branch) - using Combination4
COMB_ROOT = BASE_PATH / "Combination4_range_signal_reflec"

# Continuous 1-channel range (log-norm, 80m) saved as .npy
RANGE_ROOT = BASE_PATH / "range-normalized-continuous"

# New 4-channel dual-input dataset (B, G, R, Range)
DUAL_ROOT = BASE_PATH / "comb4-range-signal-reflec_and_range_80m"
DUAL_ROOT.mkdir(parents=True, exist_ok=True)

# Labels directory - check if exists separately
LABELS_ROOT = BASE_PATH / "labels"
if not LABELS_ROOT.exists():
    print("⚠️ Labels directory not found. You may need to download labels separately.")

print("Dataset paths:")
print(f"BASE_PATH  : {BASE_PATH}")
print(f"COMB_ROOT  : {COMB_ROOT}")
print(f"RANGE_ROOT : {RANGE_ROOT}")
print(f"DUAL_ROOT  : {DUAL_ROOT}")
print(f"LABELS_ROOT: {LABELS_ROOT}")

# Verify paths exist
for name, path in [("COMB", COMB_ROOT), ("RANGE", RANGE_ROOT)]:
    if path.exists():
        # Count files in train
        train_path = path / "train"
        if train_path.exists():
            files = list(train_path.glob("*.png")) + list(train_path.glob("*.npy"))
            print(f"✅ {name}: {len(files)} files in train")
        else:
            print(f"⚠️ {name}: exists but no train folder")
    else:
        print(f"❌ {name}: does not exist")

⚠️ Labels directory not found. You may need to download labels separately.
Dataset paths:
BASE_PATH  : C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset
COMB_ROOT  : C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\Combination4_range_signal_reflec
RANGE_ROOT : C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\range-normalized-continuous
DUAL_ROOT  : C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\comb4-range-signal-reflec_and_range_80m
LABELS_ROOT: C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\labels
✅ COMB: 1367 files in train
✅ RANGE: 1367 files in train


In [5]:
# SIMPLE TEST - Run this first to verify basic functionality
def simple_test():
    """Simple test without matplotlib to verify paths and data loading"""
    print("Running simple test...")
    print("-" * 60)
    
    # Test path existence
    print("1. Checking paths:")
    print(f"   COMB_ROOT exists: {COMB_ROOT.exists()}")
    print(f"   RANGE_ROOT exists: {RANGE_ROOT.exists()}")
    
    # Test reading one file from each
    print("\n2. Testing file reading:")
    
    # Try to read one comb image
    comb_files = list((COMB_ROOT / "train").glob("*.png"))[:1]
    if comb_files:
        test_comb = comb_files[0]
        print(f"   Testing comb: {test_comb.name}")
        img = cv2.imread(str(test_comb))
        if img is not None:
            print(f"   ✅ Comb loaded: shape={img.shape}")
        else:
            print(f"   ❌ Failed to load comb image")
    
    # Try to read one npy file
    npy_files = list((RANGE_ROOT / "train").glob("*.npy"))[:1]
    if npy_files:
        test_npy = npy_files[0]
        print(f"   Testing npy: {test_npy.name}")
        arr = np.load(test_npy)
        print(f"   ✅ NPY loaded: shape={arr.shape}, dtype={arr.dtype}")
        print(f"      Range: [{arr.min():.3f}, {arr.max():.3f}]")
    
    print("\n3. Testing 4-channel creation:")
    if comb_files and npy_files:
        # Get matching files
        stem = comb_files[0].stem
        comb_path = COMB_ROOT / "train" / f"{stem}.png"
        npy_path = RANGE_ROOT / "train" / f"{stem}.npy"
        
        if comb_path.exists() and npy_path.exists():
            comb = cv2.imread(str(comb_path))
            range_arr = np.load(npy_path)
            range_img = (np.clip(range_arr, 0, 1) * 255).astype(np.uint8)
            
            # Resize if needed
            if range_img.shape != comb.shape[:2]:
                range_img = cv2.resize(range_img, (comb.shape[1], comb.shape[0]))
            
            # Stack
            rgba = np.dstack([comb, range_img])
            print(f"   ✅ 4-channel created: shape={rgba.shape}")
            
            # Save test
            test_out = DUAL_ROOT / "simple_test.png"
            cv2.imwrite(str(test_out), rgba)
            print(f"   ✅ Saved to: {test_out}")
    
    print("-" * 60)
    print("✅ Simple test completed!")
    return True

# Run simple test first
simple_test()

Running simple test...
------------------------------------------------------------
1. Checking paths:
   COMB_ROOT exists: True
   RANGE_ROOT exists: True

2. Testing file reading:
   Testing comb: image_1.png
   ✅ Comb loaded: shape=(128, 1024, 3)
   Testing npy: image_1.npy
   ✅ NPY loaded: shape=(128, 1024), dtype=float32
      Range: [0.000, 1.000]

3. Testing 4-channel creation:
   ✅ 4-channel created: shape=(128, 1024, 4)
   ✅ Saved to: C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\comb4-range-signal-reflec_and_range_80m\simple_test.png
------------------------------------------------------------
✅ Simple test completed!


True

In [6]:
# TEST PIPELINE ON SAMPLE IMAGES
# Import matplotlib at the top level to avoid kernel crashes
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend to prevent crashes
import matplotlib.pyplot as plt

def test_dual_pipeline_sample():
    """Test the dual-branch creation on 1-2 sample images"""
    
    # Get 2 sample images from train
    comb_img_dir = COMB_ROOT / "train"
    range_npy_dir = RANGE_ROOT / "train"
    
    if not comb_img_dir.exists():
        print(f"⚠️ Comb directory not found: {comb_img_dir}")
        return
    
    sample_files = sorted(comb_img_dir.glob("*.png"))[:2]
    
    if len(sample_files) == 0:
        print("⚠️ No images found")
        return
    
    print(f"Testing on {len(sample_files)} images...")
    
    for comb_path in sample_files:
        stem = comb_path.stem
        print(f"\nProcessing: {stem}")
        
        try:
            # Read comb
            comb = cv2.imread(str(comb_path), cv2.IMREAD_COLOR)
            if comb is None:
                print(f"  ❌ Could not read: {comb_path}")
                continue
            print(f"  Comb shape: {comb.shape}, dtype: {comb.dtype}")
            
            # Read range .npy
            range_npy_path = range_npy_dir / f"{stem}.npy"
            if not range_npy_path.exists():
                print(f"  ❌ Missing: {range_npy_path}")
                continue
            
            range_arr = np.load(range_npy_path)
            print(f"  Range shape: {range_arr.shape}, dtype: {range_arr.dtype}")
            print(f"  Range stats: min={range_arr.min():.3f}, max={range_arr.max():.3f}, mean={range_arr.mean():.3f}")
            
            # Convert to uint8
            range_img = (np.clip(range_arr, 0, 1) * 255).astype(np.uint8)
            
            # Resize if needed
            if range_img.shape != comb.shape[:2]:
                print(f"  Resizing range from {range_img.shape} to {comb.shape[:2]}")
                range_img = cv2.resize(range_img, (comb.shape[1], comb.shape[0]), 
                                       interpolation=cv2.INTER_NEAREST)
            
            # Stack to 4-channel
            rgba = np.dstack([comb, range_img])
            print(f"  Final RGBA shape: {rgba.shape}, dtype: {rgba.dtype}")
            
            # Save test output instead of displaying (to avoid display issues)
            test_output_path = DUAL_ROOT / f"test_{stem}.png"
            cv2.imwrite(str(test_output_path), rgba)
            print(f"  Saved test output to: {test_output_path}")
            
            print(f"  ✅ Success!")
            
        except Exception as e:
            print(f"  ❌ Error processing {stem}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print("\n" + "="*60)
    print("✅ Test completed! Pipeline is ready.")
    print("If you want to view the images, check the DUAL_ROOT folder.")
    print("="*60)

# Run test
print("Starting pipeline test...")
test_dual_pipeline_sample()

Starting pipeline test...
Testing on 2 images...

Processing: image_1
  Comb shape: (128, 1024, 3), dtype: uint8
  Range shape: (128, 1024), dtype: float32
  Range stats: min=0.000, max=1.000, mean=0.578
  Final RGBA shape: (128, 1024, 4), dtype: uint8
  Saved test output to: C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\comb4-range-signal-reflec_and_range_80m\test_image_1.png
  ✅ Success!

Processing: image_10
  Comb shape: (128, 1024, 3), dtype: uint8
  Range shape: (128, 1024), dtype: float32
  Range stats: min=0.000, max=1.000, mean=0.557
  Final RGBA shape: (128, 1024, 4), dtype: uint8
  Saved test output to: C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\comb4-range-signal-reflec_and_range_80m\test_image_10.png
  ✅ Success!

✅ Test completed! Pipeline is ready.
If you want to view the images, check the DUAL_ROOT folder.


In [7]:
# CREATE DUAL-BRANCH DATASET
def make_dual_split(split: str):
    """Create 4-channel dual-branch images"""
    comb_img_dir = COMB_ROOT / split
    range_npy_dir = RANGE_ROOT / split
    dual_img_dir = DUAL_ROOT / "images" / split
    
    dual_img_dir.mkdir(parents=True, exist_ok=True)
    
    # Try to copy labels if they exist
    if LABELS_ROOT.exists():
        src_lbl_dir = LABELS_ROOT / split
        dual_lbl_dir = DUAL_ROOT / "labels" / split
        dual_lbl_dir.mkdir(parents=True, exist_ok=True)
        
        if src_lbl_dir.exists():
            label_files = list(src_lbl_dir.glob("*.txt"))
            for lbl in label_files:
                shutil.copy2(lbl, dual_lbl_dir / lbl.name)
            print(f"[{split}] Copied {len(label_files)} label files")
        else:
            print(f"[{split}] No labels found in {src_lbl_dir}")
    else:
        print(f"[{split}] Labels directory not found - skipping labels")
    
    if not comb_img_dir.exists():
        print(f"[{split}] Warning: {comb_img_dir} not found")
        return
    
    img_files = sorted(comb_img_dir.glob("*.png"))
    print(f"[{split}] Found {len(img_files)} comb images")
    
    processed = 0
    skipped = 0
    
    for comb_path in tqdm(img_files, desc=f"Processing {split}"):
        stem = comb_path.stem
        
        # Read comb (3ch BGR)
        comb = cv2.imread(str(comb_path), cv2.IMREAD_COLOR)
        if comb is None:
            skipped += 1
            continue
        
        # Read range .npy
        range_npy_path = range_npy_dir / f"{stem}.npy"
        if not range_npy_path.exists():
            skipped += 1
            continue
        
        range_arr = np.load(range_npy_path)
        if range_arr.ndim == 3:
            range_arr = np.squeeze(range_arr)
        
        # Convert to uint8
        range_img = (np.clip(range_arr, 0, 1) * 255).astype(np.uint8)
        
        # Resize if needed
        if range_img.shape != comb.shape[:2]:
            range_img = cv2.resize(range_img, (comb.shape[1], comb.shape[0]),
                                   interpolation=cv2.INTER_NEAREST)
        
        # Stack to 4-channel
        rgba = np.dstack([comb, range_img])
        
        out_path = dual_img_dir / f"{stem}.png"
        cv2.imwrite(str(out_path), rgba)
        processed += 1
    
    print(f"[{split}] Processed {processed} images, skipped {skipped}")

# Process all splits
print("\nCreating dual-branch dataset...")
print("="*60)
for split in ["train", "valid", "test"]:
    make_dual_split(split)
    print()


Creating dual-branch dataset...
[train] Labels directory not found - skipping labels
[train] Found 1367 comb images


Processing train: 100%|██████████| 1367/1367 [02:30<00:00,  9.08it/s]


[train] Processed 1367 images, skipped 0

[valid] Labels directory not found - skipping labels
[valid] Found 390 comb images


Processing valid: 100%|██████████| 390/390 [00:38<00:00, 10.16it/s]


[valid] Processed 390 images, skipped 0

[test] Labels directory not found - skipping labels
[test] Found 197 comb images


Processing test: 100%|██████████| 197/197 [00:23<00:00,  8.43it/s]

[test] Processed 197 images, skipped 0



In [8]:
# MODIFY YOLOV9T FOR 4-CHANNEL INPUT
# Download model if needed
if not Path("yolov9t.pt").exists():
    print("Downloading YOLOv9t...")
    import urllib.request
    urllib.request.urlretrieve(
        "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov9t.pt",
        "yolov9t.pt"
    )

model = YOLO("yolov9t.pt")
model.model.eval()

first_conv = model.model.model[0]
old_conv = first_conv.conv

# Create 4-channel conv
new_conv = torch.nn.Conv2d(
    in_channels=4,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=old_conv.bias is not None
)

# Transfer weights
with torch.no_grad():
    new_conv.weight[:, :3] = old_conv.weight
    new_conv.weight[:, 3:] = old_conv.weight[:, :1] * 0.1
    if old_conv.bias is not None:
        new_conv.bias = old_conv.bias

first_conv.conv = new_conv
model.model.model[0] = first_conv

# Save model
model.save("yolov9t_dual_4ch.pt")
print("✅ Created yolov9t_dual_4ch.pt")

✅ Created yolov9t_dual_4ch.pt


In [9]:
# CREATE DATA.YAML
DUAL_DATA_YAML = DUAL_ROOT / "data.yaml"

cfg = {
    "path": str(DUAL_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": ["snow_pole"],
    "channels": 4  # 4-channel input
}

with open(DUAL_DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

print("✅ Created data.yaml")
print(DUAL_DATA_YAML.read_text())

✅ Created data.yaml
channels: 4
names:
- snow_pole
nc: 1
path: C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\comb4-range-signal-reflec_and_range_80m
test: images/test
train: images/train
val: images/val



In [10]:
# TRAIN MODEL
# Update device based on your system (0 for GPU, 'cpu' for CPU)
device = 0 if torch.cuda.is_available() else 'cpu'

# Build command
cmd = f"""yolo train model=yolov9t_dual_4ch.pt data={DUAL_DATA_YAML} epochs=100 imgsz=1024 device={device} batch=8 patience=20 name=dual_v9t_desktop project=runs"""

print(f"Training command: {cmd}")
# Run training (uncomment to execute)
# os.system(cmd)

Training command: yolo train model=yolov9t_dual_4ch.pt data=C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\comb4-range-signal-reflec_and_range_80m\data.yaml epochs=100 imgsz=1024 device=cpu batch=8 patience=20 name=dual_v9t_desktop project=runs


In [11]:
# VALIDATE MODEL
device = 0 if torch.cuda.is_available() else 'cpu'

# Build validation command
cmd = f"""yolo val model=runs/dual_v9t_desktop/weights/best.pt data={DUAL_DATA_YAML} split=test imgsz=1024 device={device} batch=8"""

print(f"Validation command: {cmd}")
# Run validation (uncomment to execute)
# os.system(cmd)

Validation command: yolo val model=runs/dual_v9t_desktop/weights/best.pt data=C:\Users\muham\OneDrive - TU Eindhoven\Extended-evaluation-snowpole-lidar-dataset\SnowPole_Detection_Dataset\comb4-range-signal-reflec_and_range_80m\data.yaml split=test imgsz=1024 device=cpu batch=8
